# DDoS / DoS Attack Detection — CICIoMT2024 (Google Colab)

## SMOTE Ratio Experiments · XGBoost · Decision Tree · AdaBoost · Random Forest

| Item | Detail |
|------|--------|
| **Platform** | Google Colab — **Runtime → Run all** |
| **GPU** | **Runtime → Change runtime type → T4 GPU** (recommended) |
| **Dataset** | Auto-download from Kaggle via Colab Secrets |
| **Kaggle auth** | Secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY` |
| **Pipeline** | Preprocess → Train/Test split → **Resample train only** → Train → Evaluate test |
| **SMOTE ratios** | 10:1 · 5:1 · 2:1 · 1:1 (Class 0 : Class 1) |
| **Output** | `/content/outputs/` and `/content/models/` |

> **Binary label:** `1` if label contains `ddos` or `dos`, else `0`.
> Test set is **never** resampled.


---
## Cell 1 — GPU Check & Install Packages


In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU detected (XGBoost will use CUDA when supported):')
    for line in result.stdout.split('\n'):
        if any(k in line for k in ('NVIDIA', 'Tesla', 'GeForce')):
            print(' ', line.strip())
else:
    print('No GPU — enable Runtime > Change runtime type > T4 GPU.')

!pip install -q --upgrade xgboost scikit-learn imbalanced-learn psutil joblib kaggle kagglehub

import xgboost as xgb
import sklearn
print(f'\nXGBoost : {xgb.__version__}')
print(f'Sklearn : {sklearn.__version__}')
print('Packages ready.')


---
## Cell 2 — Kaggle Auth & Dataset Download

1. Open Colab **Secrets** (sidebar key icon).
2. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` from [kaggle.com/settings](https://www.kaggle.com/settings).
3. Enable **Notebook access** for both secrets.
4. Re-run this cell if download fails after adding secrets.


In [ ]:
import json, os, subprocess
from pathlib import Path

KAGGLE_DATASET = 'limamateus/cic-iomt-2024-wifi-mqtt'
DATA_DIR  = Path('/content/ciciomt2024')
MODEL_DIR = Path('/content/models')
OUTPUT_DIR = Path('/content/outputs')
for p in (DATA_DIR, MODEL_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_train.csv'
TEST_FILE  = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_test.csv'


def setup_kaggle_credentials():
    kaggle_dir = Path('/root/.kaggle')
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists():
        print('Using existing /root/.kaggle/kaggle.json')
        return

    username, key = None, None
    try:
        from google.colab import userdata
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
        print('Kaggle credentials loaded from Colab Secrets.')
    except Exception:
        username = os.environ.get('KAGGLE_USERNAME')
        key = os.environ.get('KAGGLE_KEY')
        if username and key:
            print('Kaggle credentials loaded from environment variables.')

    if not username or not key:
        raise RuntimeError(
            'Kaggle credentials not found.\n'
            'Colab sidebar > Secrets > add KAGGLE_USERNAME and KAGGLE_KEY, '
            'then enable Notebook access.'
        )

    kaggle_dir.mkdir(parents=True, exist_ok=True)
    with open(kaggle_json, 'w', encoding='utf-8') as f:
        json.dump({'username': username, 'key': key}, f)
    os.chmod(kaggle_json, 0o600)
    print('kaggle.json created.')


def ensure_dataset(require_test=True):
    global TRAIN_FILE, TEST_FILE
    if TRAIN_FILE.exists() and (not require_test or TEST_FILE.exists()):
        print(f'Dataset already present in {DATA_DIR}')
        return

    setup_kaggle_credentials()
    print(f'Downloading {KAGGLE_DATASET} ...')
    try:
        import kagglehub
        dl_path = kagglehub.dataset_download(KAGGLE_DATASET)
        print(f'kagglehub path: {dl_path}')
        for csv in Path(dl_path).rglob('*.csv'):
            target = DATA_DIR / csv.name
            if not target.exists():
                target.write_bytes(csv.read_bytes())
    except Exception as e1:
        print(f'kagglehub failed ({e1}), trying kaggle CLI...')
        subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET,
             '-p', str(DATA_DIR), '--unzip', '-q'],
            check=False,
        )

    if not TRAIN_FILE.exists():
        found = list(DATA_DIR.rglob('*train*.csv'))
        if found:
            TRAIN_FILE = found[0]
        else:
            raise FileNotFoundError(f'Train CSV not found under {DATA_DIR}')
    if require_test and not TEST_FILE.exists():
        found = list(DATA_DIR.rglob('*test*.csv'))
        if found:
            TEST_FILE = found[0]
        else:
            raise FileNotFoundError(f'Test CSV not found under {DATA_DIR}')

    print(f'\nTrain: {TRAIN_FILE}')
    if require_test:
        print(f'Test : {TEST_FILE}')


ensure_dataset(require_test=True)


---
## Cell 2 — Imports & Project Paths

Paths resolve relative to the project root (`data/`, `models/`, `outputs/`).
Override with env vars: `DATA_DIR`, `MODEL_DIR`, `OUTPUT_DIR`.


In [ ]:
# ── Step 1: Standard library & data-science imports ─────────────────────────
import os, sys, json, time, warnings, tracemalloc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from sklearn.base import clone
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_fscore_support,
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

try:
    from IPython.display import display
except ImportError:
    display = print

# ── Step 2: Resolve project paths (IDEA: cwd is usually src/ or project root) ─
DATA_DIR  = Path('/content/ciciomt2024')
MODEL_DIR = Path('/content/models')
OUTPUT_DIR = Path('/content/outputs')
for p in (DATA_DIR, MODEL_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Colab paths set in Cell 2 (Kaggle download)

TRAIN_FILE = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_train.csv'
TEST_FILE  = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_test.csv'

RANDOM_SEED = 42
MAX_ROWS = int(os.getenv('MAX_ROWS', '500000'))
TOP_N = 20
SMOTE_RATIOS = ["10:1", "5:1", "2:1", "1:1"]  # Class 0 : Class 1
EARLY_STOPPING_ROUNDS = 25   # shared patience — matches XGBoost
SKLEARN_MAX_ESTIMATORS = 300   # upper cap for RF / AdaBoost warm_start search
CV_FOLDS = int(os.getenv('CV_FOLDS', '3'))
CV_MAX_ROWS = int(os.getenv('CV_MAX_ROWS', '150000'))
np.random.seed(RANDOM_SEED)

try:
    XGBClassifier(device='cuda', n_estimators=1)
    XGB_DEVICE = 'cuda'
    print('XGBoost will use GPU (device=cuda)')
except Exception:
    XGB_DEVICE = 'cpu'
    print('XGBoost will use CPU')

print(f'Data dir   : {DATA_DIR}')
print(f'Model dir   : {MODEL_DIR}')
print(f'Output dir  : {OUTPUT_DIR}')


---
## Cell 3 — Load Dataset

Uses CSVs from `data/` when present. If missing, attempts Kaggle download
(env vars `KAGGLE_USERNAME` / `KAGGLE_KEY`, `~/.kaggle/kaggle.json`, or Colab Secrets).


In [ ]:
def load_with_label(filepath):
    """Load a CSV and normalize its label column to 'label'."""
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()
    existing = [c for c in df.columns if c.lower() in ('label', 'class', 'attack', 'type')]
    if existing:
        df.rename(columns={existing[0]: 'label'}, inplace=True)
    else:
        stem = Path(filepath).stem
        for sfx in ('_train', '_test', '-train', '-test'):
            stem = stem.replace(sfx, '')
        df['label'] = stem
    df['label'] = df['label'].astype(str).str.strip()
    return df

print(f'\nTrain: {TRAIN_FILE}')
print(f'Test : {TEST_FILE}')

print('Loading TRAIN...')
df_train = load_with_label(TRAIN_FILE)
print(f'  Train: {len(df_train):,} rows x {df_train.shape[1]} cols')

df_test = None
if TEST_FILE.exists():
    print('Loading TEST...')
    df_test = load_with_label(TEST_FILE)
    common = sorted(set(df_train.columns) & set(df_test.columns))
    df_train = df_train[common]
    df_test = df_test[common]
    print(f'  Test : {len(df_test):,} rows x {df_test.shape[1]} cols')
else:
    print('No official test file — an 80/20 split will be used later.')


---
## Cell 4 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Step 1: Basic dataset health checks ─────────────────────────────────────
print('=== DATASET OVERVIEW ===')
print(f'Rows     : {len(df_train):,}')
print(f'Columns  : {df_train.shape[1]}')
print(f'Memory   : {df_train.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'Missing  : {df_train.isnull().sum().sum():,}')
print(f'Duplicates: {df_train.duplicated().sum():,}')
print('\nDtypes:')
print(df_train.dtypes.value_counts())

# ── Step 2: Show how many rows belong to each raw attack label ───────────────
print('\n=== TOP 20 RAW LABELS ===')
label_counts = df_train['label'].value_counts()
for lbl, cnt in label_counts.head(20).items():
    print(f'  {lbl:<45} {cnt:>10,}  ({100*cnt/len(df_train):5.2f}%)')

# ── Step 3: Visualize label distribution and feature scatter sample ───────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
label_counts.head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Attack Labels (Raw)')
axes[0].set_xlabel('Count')

numeric_cols = df_train.select_dtypes(include=[np.number]).columns[:6]
if len(numeric_cols) >= 2:
    sample = df_train.sample(min(5000, len(df_train)), random_state=RANDOM_SEED)
    sns.scatterplot(data=sample, x=numeric_cols[0], y=numeric_cols[1],
                    hue='label', legend=False, alpha=0.4, ax=axes[1])
    axes[1].set_title(f'{numeric_cols[0]} vs {numeric_cols[1]} (sample)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 5 — Binary Label: DDoS or DoS vs Other

In [ ]:
# ── Step 1: Map multi-class labels to binary attack vs non-attack ────────────
def to_attack_binary(label: str) -> int:
    """Return 1 if label mentions ddos/dos; otherwise 0 (benign/other attacks)."""
    text = str(label).lower()
    return 1 if ('ddos' in text or 'dos' in text) else 0

df_train['binary_label'] = df_train['label'].apply(to_attack_binary)
if df_test is not None:
    df_test['binary_label'] = df_test['label'].apply(to_attack_binary)

# ── Step 2: Report class balance on training data ───────────────────────────
counts = df_train['binary_label'].value_counts().sort_index()
print('Binary distribution (train):')
print(f'  Other (0) : {counts.get(0, 0):,}')
print(f'  DDoS/DoS(1): {counts.get(1, 0):,}')
print(f'  Imbalance : {counts.max()/max(counts.min(),1):.1f}x')

# ── Step 3: Plot binary distribution for the report ───────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_xticklabels(['Other (0)', 'DDoS/DoS (1)'], rotation=0)
ax.set_title('Binary Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'binary_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 6 — Preprocessing Pipeline (no resampling)


In [ ]:
# Columns that are identifiers/labels — never used as model features
DROP_COLS = ['label', 'binary_label', 'flow_id', 'Flow ID',
             'src_ip', 'Src IP', 'dst_ip', 'Dst IP']

def get_Xy(df, target='binary_label'):
    """Split dataframe into feature matrix X and target vector y."""
    drop = [c for c in DROP_COLS if c in df.columns]
    X = df.drop(columns=drop).copy()
    y = df[target].copy().reset_index(drop=True)
    return X, y

# ── Step 1: Extract features/target from training dataframe ─────────────────
X_raw, y = get_Xy(df_train)
y_groups = df_train['label'].astype(str).str.strip().reset_index(drop=True)

# ── Step 2: Label-encode categoricals (fit encoders on train only) ──────────
label_encoders = {}
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    X_raw[col] = le.fit_transform(X_raw[col].astype(str))
    label_encoders[col] = le

# ── Step 3: Clean infinities/NaNs; drop mostly-empty columns; impute medians
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
miss_pct = X_raw.isnull().mean()
drop_miss = miss_pct[miss_pct > 0.5].index.tolist()
if drop_miss:
    X_raw.drop(columns=drop_miss, inplace=True)
train_medians = X_raw.median(numeric_only=True)
X_raw.fillna(train_medians, inplace=True)
X_raw = X_raw.select_dtypes(include=[np.number]).reset_index(drop=True)
feature_columns = X_raw.columns.tolist()
print(f'Numeric features: {len(feature_columns)}')

# ── Step 4: Remove exact duplicate feature rows ─────────────────────────────
mask = ~X_raw.duplicated()
X_raw, y = X_raw[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
y_groups = y_groups[mask].reset_index(drop=True)
print(f'After dedup: {len(X_raw):,} rows')

# ── Step 5: Stratified down-sample if dataset exceeds MAX_ROWS ───────────────
if len(X_raw) > MAX_ROWS:
    print(f'Sampling {MAX_ROWS:,} from {len(X_raw):,} rows...')
    rng = np.random.default_rng(RANDOM_SEED)
    parts = []
    for cls in y.unique():
        idx = y[y == cls].index.to_numpy()
        n_take = int(MAX_ROWS * len(idx) / len(y))
        n_take = min(n_take, len(idx))
        parts.append(rng.choice(idx, n_take, replace=False))
    sampled_idx = np.concatenate(parts)
    rng.shuffle(sampled_idx)
    X_raw = X_raw.iloc[sampled_idx].reset_index(drop=True)
    y = y.iloc[sampled_idx].reset_index(drop=True)
    y_groups = y_groups.iloc[sampled_idx].reset_index(drop=True)
print(f'Working set: {len(X_raw):,} rows')

# ── Step 6: Standardize features (fit scaler on train only) ─────────────────
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)

# ── Step 7: Build train/test matrices (official split or 80/20 stratified) ───
if df_test is not None:
    X_test_raw, y_test = get_Xy(df_test)
    for col, le in label_encoders.items():
        if col in X_test_raw.columns:
            X_test_raw[col] = (
                X_test_raw[col].astype(str)
                .apply(lambda v: int(le.transform([v])[0]) if v in le.classes_ else 0)
            )
    X_test_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_test_raw.fillna(train_medians, inplace=True)
    X_test_raw = X_test_raw[[c for c in feature_columns if c in X_test_raw.columns]]
    X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=X_test_raw.columns)
    X_train, y_train = X_scaled, y
    label_groups_train = y_groups.reset_index(drop=True)
    print('Using official CIC train/test split.')
else:
    train_idx, test_idx = train_test_split(
        np.arange(len(X_scaled)), test_size=0.2, random_state=RANDOM_SEED, stratify=y)
    X_train = X_scaled.iloc[train_idx].reset_index(drop=True)
    X_test  = X_scaled.iloc[test_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_test  = y.iloc[test_idx].reset_index(drop=True)
    label_groups_train = y_groups.iloc[train_idx].reset_index(drop=True)
    print('Using 80/20 stratified split.')


# ── Step 8: Leakage guard — drop train rows identical to official test flows ──
if df_test is not None:
    common_feat = [c for c in feature_columns if c in X_test_raw.columns]
    train_sig = pd.util.hash_pandas_object(
        X_raw[common_feat].fillna(-999).astype(str), index=False)
    test_sig = pd.util.hash_pandas_object(
        X_test_raw[common_feat].fillna(-999).astype(str), index=False)
    overlap_mask = train_sig.isin(set(test_sig))
    n_overlap = int(overlap_mask.sum())
    if n_overlap:
        print(f'Removing {n_overlap:,} train rows that duplicate official test flows (leakage guard).')
        keep = ~overlap_mask.to_numpy()
        X_raw = X_raw.loc[keep].reset_index(drop=True)
        y = y.loc[keep].reset_index(drop=True)
        y_groups = y_groups.loc[keep].reset_index(drop=True)
        X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)
        X_train, y_train = X_scaled, y
        label_groups_train = y_groups.reset_index(drop=True)
    else:
        print('Train/test feature-vector overlap: 0 rows (no cross-split duplicates).')

# ── Step 9: Report class balance (NO resampling here — done per experiment) ───
neg_train = int((y_train == 0).sum())
pos_train = int((y_train == 1).sum())
XGB_SCALE_POS_WEIGHT = neg_train / max(pos_train, 1)
majority_baseline = max(y_test.value_counts()) / len(y_test)
train_ratio = neg_train / max(pos_train, 1)
print(f'Train class counts — Other(0): {neg_train:,} | DDoS/DoS(1): {pos_train:,} | ratio {train_ratio:.2f}:1')
print(f'XGB scale_pos_weight (pre-resample train): {XGB_SCALE_POS_WEIGHT:.4f}')
print(f'Majority-class baseline accuracy (test): {majority_baseline:.4f}')
print(f'\nTrain: {X_train.shape} | Test (untouched): {X_test.shape}')
print('Resampling (SMOTE / undersampling) is applied inside each experiment on TRAIN only.')


---
## Cell 7 — Evaluation Utilities


In [ ]:
# Global store for each model's metrics and predictions
ALL_RESULTS = {}

def _score_split(y_true, y_pred, y_prob):
    """Compute classification metrics and false-positive rate for one split."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1-Score': round(f1_score(y_true, y_pred, zero_division=0), 4),
        'FPR': round(fp / (fp + tn + 1e-9), 4),
        'AUC': round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else None,
    }


def measure_model(name, model, X_tr, y_tr, X_te, y_te, fit_fn=None,
                  X_val=None, y_val=None, eval_train=True):
    """Train, evaluate on held-out test, and optionally on real (pre-SMOTE) train."""
    tracemalloc.start()
    proc = psutil.Process()
    mem_before = proc.memory_info().rss / 1e6

    # ── Train and time fit phase ────────────────────────────────────────────
    t0 = time.time()
    if fit_fn:
        fit_fn(model, X_tr, y_tr)
    else:
        model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    # ── Predict on test and time inference phase ────────────────────────────
    t1 = time.time()
    y_pred = model.predict(X_te)
    infer_time = time.time() - t1
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
    tracemalloc.stop()
    rss_after = proc.memory_info().rss / 1e6

    test_scores = _score_split(y_te, y_pred, y_prob)
    total_time = train_time + infer_time
    efficiency = test_scores['F1-Score'] / total_time if total_time > 0 else 0

    row = {
        'Model': name,
        **test_scores,
        'Train_Time_s': round(train_time, 2),
        'Infer_Time_s': round(infer_time, 4),
        'Peak_Mem_MB': round(peak_mem, 1),
        'RSS_Delta_MB': round(rss_after - mem_before, 1),
        'Efficiency': round(efficiency, 4),
        'n_features': X_tr.shape[1],
    }

    # ── Optional train-set evaluation (detect overfitting vs test) ──────────
    if eval_train and X_val is not None and y_val is not None:
        tr_pred = model.predict(X_val)
        tr_prob = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else None
        tr_scores = _score_split(y_val, tr_pred, tr_prob)
        row['Train_AUC'] = tr_scores['AUC']
        row['Train_F1'] = tr_scores['F1-Score']
        row['AUC_Gap'] = round((tr_scores['AUC'] or 0) - (test_scores['AUC'] or 0), 4)
        row['F1_Gap'] = round(tr_scores['F1-Score'] - test_scores['F1-Score'], 4)

    print(f'\n{"="*60}\n  {name}\n{"="*60}')
    for k in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'FPR', 'AUC']:
        if row.get(k) is not None:
            print(f'  {k:<12}: {row[k]}')
    if 'Train_AUC' in row:
        print(f'  Train AUC   : {row["Train_AUC"]}  (real train, pre-SMOTE)')
        print(f'  AUC gap     : {row["AUC_Gap"]}  (train - test; large => overfit)')
    print(f'  Train time  : {train_time:.2f}s')
    print(f'  Infer time  : {infer_time:.4f}s')
    print(f'  Peak memory : {peak_mem:.1f} MB')
    print(classification_report(y_te, y_pred, target_names=['Other', 'DDoS/DoS'], zero_division=0))
    return model, y_pred, y_prob, row

---
## Cell 8 — Resampling Helpers & Feature Selection


In [ ]:
# ── Resampling helpers (train only, after train/test split) ─────────────────

def _parse_ratio(ratio_label: str):
    """Parse '10:1' as Class0:Class1 target counts ratio."""
    parts = ratio_label.strip().split(':')
    if len(parts) != 2:
        raise ValueError(f'Expected Class0:Class1 format, got {ratio_label!r}')
    return int(parts[0]), int(parts[1])


def resample_train_to_ratio(X_train, y_train, ratio_label, random_state=RANDOM_SEED):
    """
    Resample training data to target Class0:Class1 ratio using SMOTE or undersampling.

    Pipeline: split first → resample train only → model fit → evaluate on untouched test.
    Uses dict sampling_strategy (absolute counts) — imblearn floats only allow (0, 1].
    """
    r0, r1 = _parse_ratio(ratio_label)
    target_ratio = r0 / r1  # n0 / n1

    n0 = int((y_train == 0).sum())
    n1 = int((y_train == 1).sum())
    current_ratio = n0 / max(n1, 1)

    if abs(current_ratio - target_ratio) / max(target_ratio, 1e-9) < 1e-6:
        return X_train.copy(), y_train.copy(), 'none'

    # Target counts for Class0:Class1 = r0:r1
    n0_if_keep_n1 = max(1, int(round(n1 * target_ratio)))
    n1_if_keep_n0 = max(1, int(round(n0 * r1 / r0)))

    if current_ratio > target_ratio:
        # Too much class 0 — undersample class 0 or SMOTE class 1
        if n0 > n0_if_keep_n1:
            strategy = {0: n0_if_keep_n1}
            sampler = RandomUnderSampler(sampling_strategy=strategy, random_state=random_state)
            method = 'undersample'
        else:
            strategy = {1: n1_if_keep_n0}
            k = min(5, n1 - 1)
            if k < 1:
                print(f'  Warning: too few class-1 samples for {ratio_label}; skipping resample.')
                return X_train.copy(), y_train.copy(), 'skipped'
            sampler = SMOTE(sampling_strategy=strategy, random_state=random_state, k_neighbors=k)
            method = 'SMOTE'
    else:
        # Too little class 0 — SMOTE class 0 or undersample class 1
        if n0 < n0_if_keep_n1:
            strategy = {0: n0_if_keep_n1}
            k = min(5, n0 - 1)
            if k < 1:
                print(f'  Warning: too few class-0 samples for {ratio_label}; skipping resample.')
                return X_train.copy(), y_train.copy(), 'skipped'
            sampler = SMOTE(sampling_strategy=strategy, random_state=random_state, k_neighbors=k)
            method = 'SMOTE'
        else:
            strategy = {1: n1_if_keep_n0}
            sampler = RandomUnderSampler(sampling_strategy=strategy, random_state=random_state)
            method = 'undersample'

    X_res, y_res = sampler.fit_resample(X_train, y_train)
    X_res = pd.DataFrame(X_res, columns=X_train.columns)
    y_res = pd.Series(y_res, name='binary_label').reset_index(drop=True)

    n0r = int((y_res == 0).sum())
    n1r = int((y_res == 1).sum())
    print(f'  {method} -> Other(0)={n0r:,} DDoS/DoS(1)={n1r:,} ({n0r/max(n1r,1):.2f}:1)')
    return X_res, y_res, method


def select_features_xgb(X_fit, y_fit, X_eval, scale_pos_weight):
    """XGB gain-importance top-N on resampled train; apply to train and test."""
    xgb_sel = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.2,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        device=XGB_DEVICE, random_state=RANDOM_SEED, verbosity=0,
        eval_metric='logloss')
    xgb_sel.fit(X_fit, y_fit)
    features = (
        pd.DataFrame({'feature': X_fit.columns, 'importance': xgb_sel.feature_importances_})
        .sort_values('importance', ascending=False)
        .head(TOP_N)['feature'].tolist()
    )
    return features, xgb_sel


def per_class_recall(y_true, y_pred):
    """Return recall for Class 0 (Other) and Class 1 (DDoS/DoS)."""
    rec = recall_score(y_true, y_pred, labels=[0, 1], average=None, zero_division=0)
    return round(float(rec[0]), 4), round(float(rec[1]), 4)


def fit_xgb(m, X, y):
    """Train XGBoost with a small internal validation set for early stopping."""
    Xf, Xv, yf, yv = train_test_split(
        X, y, test_size=0.1, random_state=RANDOM_SEED, stratify=y)
    m.fit(Xf, yf, eval_set=[(Xv, yv)], verbose=False)


def _train_val_split(X, y):
    """90/10 stratified split — same protocol as fit_xgb."""
    return train_test_split(
        X, y, test_size=0.1, random_state=RANDOM_SEED, stratify=y)


def fit_decision_tree(m, X, y):
    """Regularized DT — pick max_depth by validation F1 (early stopping on tree depth)."""
    Xf, Xv, yf, yv = _train_val_split(X, y)
    base_params = m.get_params()
    cap_depth = base_params.get('max_depth') or 8
    best_f1, best_depth = -1.0, 2
    rounds_no_improve = 0
    for depth in range(2, int(cap_depth) + 1):
        trial = DecisionTreeClassifier(**{**base_params, 'max_depth': depth})
        trial.fit(Xf, yf)
        f1 = f1_score(yv, trial.predict(Xv), zero_division=0)
        if f1 > best_f1:
            best_f1, best_depth = f1, depth
            rounds_no_improve = 0
        else:
            rounds_no_improve += 1
            if rounds_no_improve >= EARLY_STOPPING_ROUNDS:
                break
    m.set_params(max_depth=best_depth)
    m.fit(Xf, yf)
    print(f'    max_depth={best_depth} (val F1={best_f1:.4f})')


def fit_rf_early_stop(m, X, y, step=5, patience=None):
    """Random Forest: warm_start + validation F1 early stopping."""
    patience = patience or EARLY_STOPPING_ROUNDS
    Xf, Xv, yf, yv = _train_val_split(X, y)
    probe = clone(m)
    probe.set_params(warm_start=True, n_estimators=0)
    max_n = SKLEARN_MAX_ESTIMATORS
    best_f1, best_n = -1.0, step
    rounds_no_improve = 0
    for n in range(step, max_n + 1, step):
        probe.set_params(n_estimators=n)
        probe.fit(Xf, yf)
        f1 = f1_score(yv, probe.predict(Xv), zero_division=0)
        if f1 > best_f1:
            best_f1, best_n = f1, n
            rounds_no_improve = 0
        else:
            rounds_no_improve += 1
            if rounds_no_improve >= patience:
                break
    m.set_params(n_estimators=best_n, warm_start=False)
    m.fit(Xf, yf)
    print(f'    early stop @ {best_n} estimators (val F1={best_f1:.4f})')


def fit_adaboost_early_stop(m, X, y, patience=None):
    """AdaBoost: staged_predict validation F1 early stopping (no warm_start in sklearn 1.9)."""
    patience = patience or EARLY_STOPPING_ROUNDS
    Xf, Xv, yf, yv = _train_val_split(X, y)
    probe = clone(m)
    probe.set_params(n_estimators=SKLEARN_MAX_ESTIMATORS)
    probe.fit(Xf, yf)
    best_f1, best_n = -1.0, 1
    rounds_no_improve = 0
    for n, y_pred in enumerate(probe.staged_predict(Xv), start=1):
        f1 = f1_score(yv, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_n = f1, n
            rounds_no_improve = 0
        else:
            rounds_no_improve += 1
            if rounds_no_improve >= patience:
                break
    m.set_params(n_estimators=best_n)
    m.fit(Xf, yf)
    print(f'    early stop @ {best_n} estimators (val F1={best_f1:.4f})')


---
## Cell 9 — SMOTE Ratio Experiments (All Models)

All four models share **90/10 validation early stopping** (`EARLY_STOPPING_ROUNDS=25`) and regularized hyperparameters for a fair comparison.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SMOTE ratio experiments — XGBoost, Decision Tree, AdaBoost, Random Forest
# ═══════════════════════════════════════════════════════════════════════════

EXPERIMENT_MODELS = ['XGBoost', 'Decision Tree', 'AdaBoost', 'Random Forest']

MODEL_COLORS = {
    'XGBoost': '#e74c3c',
    'Decision Tree': '#3498db',
    'AdaBoost': '#f39c12',
    'Random Forest': '#2ecc71',
}


def create_model(name, scale_pos_weight):
    """Instantiate classifiers — all models use regularization + validation early stopping."""
    if name == 'XGBoost':
        return XGBClassifier(
            n_estimators=SKLEARN_MAX_ESTIMATORS, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, gamma=0.2,
            reg_alpha=0.1, reg_lambda=2.0, min_child_weight=5,
            scale_pos_weight=scale_pos_weight, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            device=XGB_DEVICE, random_state=RANDOM_SEED, verbosity=0,
            eval_metric='logloss')
    if name == 'Decision Tree':
        return DecisionTreeClassifier(
            criterion='gini', max_depth=8, min_samples_split=50,
            min_samples_leaf=25, max_features='sqrt',
            min_impurity_decrease=1e-4, random_state=RANDOM_SEED)
    if name == 'AdaBoost':
        return AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, min_samples_leaf=25),
            n_estimators=SKLEARN_MAX_ESTIMATORS, learning_rate=0.05,
            random_state=RANDOM_SEED)
    if name == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=SKLEARN_MAX_ESTIMATORS, max_depth=8,
            min_samples_split=50, min_samples_leaf=25,
            max_features='sqrt', max_samples=0.8,
            random_state=RANDOM_SEED, n_jobs=-1)
    raise ValueError(f'Unknown model: {name}')


def train_model(name, model, X_tr, y_res):
    """Fit with validation early stopping (all models share 90/10 split + patience)."""
    t0 = time.time()
    if name == 'XGBoost':
        fit_xgb(model, X_tr, y_res)
    elif name == 'Decision Tree':
        fit_decision_tree(model, X_tr, y_res)
    elif name == 'Random Forest':
        fit_rf_early_stop(model, X_tr, y_res, step=5)
    elif name == 'AdaBoost':
        fit_adaboost_early_stop(model, X_tr, y_res)
    else:
        model.fit(X_tr, y_res)
    return time.time() - t0


SMOTE_EXPERIMENT_RESULTS = []

for ratio_label in SMOTE_RATIOS:
    print(f'\n{"="*70}\n  SMOTE ratio experiment: {ratio_label} (Class 0 : Class 1)\n{"="*70}')

    # Train only → resample → feature selection → train each model → evaluate on untouched test
    X_res, y_res, resample_method = resample_train_to_ratio(X_train, y_train, ratio_label)

    spw = 1.0  # SMOTE already set train ratio — do not double-apply pre-SMOTE weight
    selected_features, _ = select_features_xgb(X_res, y_res, X_test, spw)
    X_tr = X_res[selected_features]
    X_te = X_test[selected_features]

    for model_name in EXPERIMENT_MODELS:
        print(f'\n  --- {model_name} ---')
        model = create_model(model_name, spw)
        train_time = train_model(model_name, model, X_tr, y_res)

        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]
        rec0, rec1 = per_class_recall(y_test, y_pred)
        acc = round(accuracy_score(y_test, y_pred), 4)
        auc = round(roc_auc_score(y_test, y_prob), 4)
        f1 = round(f1_score(y_test, y_pred, zero_division=0), 4)

        row = {
            'Model': model_name,
            'SMOTE_Ratio': ratio_label,
            'Resample_Method': resample_method,
            'Train_Rows_After_Resample': len(X_res),
            'Class_0_Recall': rec0,
            'Class_1_Recall': rec1,
            'Accuracy': acc,
            'F1-Score': f1,
            'AUC': auc,
            'Train_Time_s': round(train_time, 2),
            'n_features': len(selected_features),
        }
        SMOTE_EXPERIMENT_RESULTS.append(row)

        print(f'  Class 0 Recall (Other)    : {rec0}')
        print(f'  Class 1 Recall (DDoS/DoS) : {rec1}')
        print(f'  Accuracy                  : {acc}')
        print(f'  F1-Score                  : {f1}')
        print(f'  AUC                       : {auc}')
        print(f'  Train time                : {train_time:.2f}s')

smote_results_df = pd.DataFrame(SMOTE_EXPERIMENT_RESULTS)
print('\n=== SMOTE RATIO EXPERIMENT SUMMARY (ALL MODELS) ===')
display(smote_results_df)
smote_results_df.to_csv(OUTPUT_DIR / 'smote_ratio_experiments.csv', index=False)
print(f'Saved: {OUTPUT_DIR / "smote_ratio_experiments.csv"}')


---
## Cell 10 — Results Visualization


In [ ]:
# ── Visualize SMOTE ratio experiment results (all models) ─────────────────
ratio_order = SMOTE_RATIOS
model_order = EXPERIMENT_MODELS
n_ratios = len(ratio_order)
n_models = len(model_order)
x = np.arange(n_ratios)
width = 0.18

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, model_name in enumerate(model_order):
    sub = smote_results_df[smote_results_df['Model'] == model_name].set_index('SMOTE_Ratio').reindex(ratio_order)
    offset = (i - (n_models - 1) / 2) * width
    color = MODEL_COLORS[model_name]
    axes[0].bar(x + offset, sub['Class_1_Recall'], width, label=model_name, color=color)
    axes[1].bar(x + offset, sub['Accuracy'], width, label=model_name, color=color)
    axes[2].bar(x + offset, sub['AUC'], width, label=model_name, color=color)

for ax, title, ylabel in zip(
    axes,
    ['Class 1 Recall (DDoS/DoS) by SMOTE Ratio', 'Accuracy by SMOTE Ratio', 'AUC by SMOTE Ratio'],
    ['Recall', 'Accuracy', 'AUC'],
):
    ax.set_xticks(x)
    ax.set_xticklabels(ratio_order)
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.set_xlabel('Target ratio (Class 0 : Class 1)')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'smote_ratio_experiments.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR / "smote_ratio_experiments.png"}')

# ── Heatmap: Class 1 recall (rows=models, cols=ratios) ─────────────────────
heat = smote_results_df.pivot(index='Model', columns='SMOTE_Ratio', values='Class_1_Recall')
heat = heat.reindex(index=model_order, columns=ratio_order)
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heat, annot=True, fmt='.4f', cmap='YlOrRd', vmin=0.9, vmax=1.0, ax=ax)
ax.set_title('Class 1 Recall (DDoS/DoS) — Model × SMOTE Ratio')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'smote_ratio_class1_recall_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR / "smote_ratio_class1_recall_heatmap.png"}')


<!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed -->

<!-- <!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed --> -->
<!--  -->
<!-- <!-- --- --> -->
<!-- <!-- ## Cell 12 — GroupKFold by Attack Label --> -->


In [ ]:
# ── Original v2 cell (commented out) — restore when needed ──
# # ── Original v2 cell (commented out) — restore when needed ──
# # def _subset_cv_data(X, y, groups, max_rows):
# #     """Keep whole attack-label groups until roughly max_rows is reached."""
# #     if len(X) <= max_rows:
# #         return X.reset_index(drop=True), y.reset_index(drop=True), groups.reset_index(drop=True)
# #
# #     rng = np.random.default_rng(RANDOM_SEED)
# #     group_counts = groups.value_counts()
# #     shuffled_groups = rng.permutation(group_counts.index.to_numpy())
# #     picked, row_count = [], 0
# #     for grp in shuffled_groups:
# #         picked.append(grp)
# #         row_count += int(group_counts[grp])
# #         if row_count >= max_rows:
# #             break
# #     mask = groups.isin(picked)
# #     return (
# #         X.loc[mask].reset_index(drop=True),
# #         y.loc[mask].reset_index(drop=True),
# #         groups.loc[mask].reset_index(drop=True),
# #     )
# #
# #
# # def _fold_metrics(y_true, y_pred, y_prob):
# #     """Macro-F1, per-class recall, AUC, and FPR for one validation fold."""
# #     prec, rec, _, _ = precision_recall_fscore_support(
# #         y_true, y_pred, labels=[0, 1], zero_division=0)
# #     cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
# #     tn, fp, fn, tp = cm.ravel()
# #     return {
# #         'Accuracy': round(accuracy_score(y_true, y_pred), 4),
# #         'Macro_F1': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
# #         'Recall_Other': round(rec[0], 4),
# #         'Recall_DDoS_DoS': round(rec[1], 4),
# #         'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
# #         'FPR': round(fp / (fp + tn + 1e-9), 4),
# #         'AUC': round(roc_auc_score(y_true, y_prob), 4) if len(np.unique(y_true)) > 1 else None,
# #     }
# #
# #
# # def _fit_fold_pipeline(X_tr, y_tr, X_va, y_va, model_name):
# #     """Leakage-safe fold pipeline: SMOTE + per-fold XGB feature selection + model fit."""
# #     neg = int((y_tr == 0).sum())
# #     pos = int((y_tr == 1).sum())
# #     fold_ratio = max(neg, pos) / max(min(neg, pos), 1)
# #
# #     if fold_ratio > 1.5 and min(neg, pos) > 1:
# #         k = min(5, min(neg, pos) - 1)
# #         X_fit, y_fit = SMOTE(random_state=RANDOM_SEED, k_neighbors=k).fit_resample(X_tr, y_tr)
# #         X_fit = pd.DataFrame(X_fit, columns=X_tr.columns)
# #         y_fit = pd.Series(y_fit, name='binary_label')
# #     else:
# #         X_fit, y_fit = X_tr.copy(), y_tr.copy()
# #
# #     spw = neg / max(pos, 1)
# #     xgb_sel = XGBClassifier(
# #         n_estimators=100, max_depth=6, learning_rate=0.2,
# #         subsample=0.8, colsample_bytree=0.8,
# #         scale_pos_weight=spw, device=XGB_DEVICE,
# #         random_state=RANDOM_SEED, verbosity=0, eval_metric='logloss')
# #     xgb_sel.fit(X_fit, y_fit)
# #
# #     fold_features = (
# #         pd.DataFrame({'feature': X_fit.columns, 'importance': xgb_sel.feature_importances_})
# #         .sort_values('importance', ascending=False)
# #         .head(TOP_N)['feature'].tolist()
# #     )
# #
# #     if model_name == 'XGBoost':
# #         model = XGBClassifier(
# #             n_estimators=300, max_depth=6, learning_rate=0.05,
# #             subsample=0.8, colsample_bytree=0.8,
# #             reg_alpha=0.1, reg_lambda=1.0,
# #             scale_pos_weight=spw, device=XGB_DEVICE,
# #             random_state=RANDOM_SEED, verbosity=0, eval_metric='logloss')
# #     elif model_name == 'Random Forest':
# #         model = RandomForestClassifier(
# #             n_estimators=200, max_depth=12, min_samples_leaf=10,
# #             n_jobs=-1, random_state=RANDOM_SEED)
# #     else:
# #         model = DecisionTreeClassifier(
# #             max_depth=12, min_samples_split=20, min_samples_leaf=10,
# #             random_state=RANDOM_SEED)
# #
# #     X_fit_sel = X_fit[fold_features]
# #     X_va_sel = X_va[fold_features]
# #     model.fit(X_fit_sel, y_fit)
# #     y_pred = model.predict(X_va_sel)
# #     y_prob = model.predict_proba(X_va_sel)[:, 1]
# #     return _fold_metrics(y_va, y_pred, y_prob), len(fold_features)
# #
# #
# # # ── Step 1: Prepare CV subset (train only — never uses official test) ───────
# # X_cv, y_cv, groups_cv = _subset_cv_data(
# #     X_train, y_train, label_groups_train, CV_MAX_ROWS)
# # n_groups = groups_cv.nunique()
# # n_splits = min(CV_FOLDS, n_groups)
# # if n_splits < 2:
# #     raise ValueError(f'Need at least 2 attack-label groups for CV, found {n_groups}.')
# #
# # print('=== GROUPKFOLD BY ATTACK LABEL ===')
# # print(f'CV rows: {len(X_cv):,} | attack-label groups: {n_groups} | folds: {n_splits}')
# # print('Scaler/SMOTE/feature selection are re-fit inside each train fold only.')
# #
# # gkf = GroupKFold(n_splits=n_splits)
# # cv_models = ['XGBoost', 'Random Forest', 'Decision Tree']
# # cv_fold_rows = []
# #
# # # ── Step 2: Run leakage-safe GroupKFold for each model ──────────────────────
# # for model_name in cv_models:
# #     for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(X_cv, y_cv, groups_cv)):
# #         X_tr_fold = X_cv.iloc[tr_idx]
# #         y_tr_fold = y_cv.iloc[tr_idx]
# #         X_va_fold = X_cv.iloc[va_idx]
# #         y_va_fold = y_cv.iloc[va_idx]
# #         held_out = sorted(groups_cv.iloc[va_idx].unique())
# #
# #         metrics, n_feats = _fit_fold_pipeline(
# #             X_tr_fold, y_tr_fold, X_va_fold, y_va_fold, model_name)
# #         cv_fold_rows.append({
# #             'Algorithm': model_name,
# #             'Fold': fold_idx + 1,
# #             'Held_Out_Groups': len(held_out),
# #             'Val_Rows': len(va_idx),
# #             'Features': n_feats,
# #             **metrics,
# #         })
# #         print(
# #             f"  {model_name} fold {fold_idx + 1}: AUC={metrics['AUC']} "
# #             f"Macro-F1={metrics['Macro_F1']} "
# #             f"Recall(Other)={metrics['Recall_Other']} "
# #             f"Recall(DDoS/DoS)={metrics['Recall_DDoS_DoS']}"
# #         )
# #
# # cv_fold_df = pd.DataFrame(cv_fold_rows)
# # cv_summary_df = (
# #     cv_fold_df.groupby('Algorithm', as_index=False)
# #     .agg(
# #         Folds=('Fold', 'count'),
# #         AUC_mean=('AUC', 'mean'),
# #         AUC_std=('AUC', 'std'),
# #         Macro_F1_mean=('Macro_F1', 'mean'),
# #         Macro_F1_std=('Macro_F1', 'std'),
# #         Recall_Other_mean=('Recall_Other', 'mean'),
# #         Recall_DDoS_DoS_mean=('Recall_DDoS_DoS', 'mean'),
# #         FPR_mean=('FPR', 'mean'),
# #     )
# #     .round(4)
# # )
# #
# # # ── Step 3: Compare CV vs official held-out test ────────────────────────────
# # official_rows = []
# # for algo, bundle in ALL_RESULTS.items():
# #     m = bundle['metrics']
# #     official_rows.append({
# #         'Algorithm': algo,
# #         'Official_Test_AUC': m.get('AUC'),
# #         'Official_Test_F1': m.get('F1-Score'),
# #         'Official_Test_Accuracy': m.get('Accuracy'),
# #     })
# # official_df = pd.DataFrame(official_rows)
# #
# # cv_compare_df = cv_summary_df.merge(official_df, on='Algorithm', how='left')
# # cv_compare_df['AUC_Drop_vs_Official'] = (
# #     cv_compare_df['Official_Test_AUC'] - cv_compare_df['AUC_mean']).round(4)
# #
# # print('\n=== GROUPKFOLD SUMMARY (mean ± std across folds) ===')
# # display(cv_summary_df)
# #
# # print('\n=== CV vs OFFICIAL TEST (positive drop => official split is optimistic) ===')
# # display(cv_compare_df[['Algorithm', 'AUC_mean', 'Official_Test_AUC', 'AUC_Drop_vs_Official',
# #                        'Macro_F1_mean', 'Recall_Other_mean', 'Recall_DDoS_DoS_mean']])
# #
# # cv_fold_df.to_csv(OUTPUT_DIR / 'groupkfold_cv_results.csv', index=False)
# # cv_summary_df.to_csv(OUTPUT_DIR / 'groupkfold_cv_summary.csv', index=False)
# # print(f"\nSaved: {OUTPUT_DIR / 'groupkfold_cv_results.csv'}")
# #
# # # ── Step 4: Visualize CV vs official test AUC ───────────────────────────────
# # fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# #
# # plot_df = cv_summary_df.set_index('Algorithm')
# # x = np.arange(len(plot_df))
# # axes[0].bar(x - 0.2, plot_df['AUC_mean'], width=0.35, label='GroupKFold AUC', color='steelblue')
# # if not official_df.empty:
# #     official_map = official_df.set_index('Algorithm')['Official_Test_AUC']
# #     axes[0].bar(x + 0.2, [official_map.get(a, np.nan) for a in plot_df.index],
# #                 width=0.35, label='Official test AUC', color='coral')
# # axes[0].set_xticks(x)
# # axes[0].set_xticklabels(plot_df.index, rotation=15)
# # axes[0].set_ylim(0, 1.05)
# # axes[0].set_title('AUC: GroupKFold vs Official Test')
# # axes[0].legend()
# #
# # axes[1].bar(plot_df.index, plot_df['Macro_F1_mean'], color='seagreen')
# # axes[1].set_ylim(0, 1.05)
# # axes[1].set_title('GroupKFold Macro-F1 (per-class average)')
# # axes[1].tick_params(axis='x', rotation=15)
# #
# # plt.tight_layout()
# # plt.savefig(OUTPUT_DIR / 'groupkfold_cv.png', dpi=150, bbox_inches='tight')
# # plt.show()
# # print(f"Saved: {OUTPUT_DIR / 'groupkfold_cv.png'}")


<!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed -->

<!-- <!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed --> -->
<!--  -->
<!-- <!-- --- --> -->
<!-- <!-- ## Cell 13 — Feature List --> -->


In [ ]:
# ── Original v2 cell (commented out) — restore when needed ──
# # ── Original v2 cell (commented out) — restore when needed ──
# # # ── Step 1: Record the unified feature set used by all four models ─────────
# # MODEL_FEATURE_SETS = {
# #     name: {'method': FS_METHOD, 'features': SELECTED_FEATURES}
# #     for name in ['Decision Tree', 'XGBoost', 'AdaBoost', 'Random Forest']
# # }
# #
# # feature_sets = {name: set(info['features']) for name, info in MODEL_FEATURE_SETS.items()}
# # common_features = sorted(set.intersection(*feature_sets.values()))
# #
# # # ── Step 2: Print numbered feature list ───────────────────────────────────────
# # print('=' * 70)
# # print(f'UNIFIED FEATURES ({FS_METHOD}) — used by all 4 models')
# # print('=' * 70)
# # for i, feat in enumerate(common_features, 1):
# #     print(f'  {i:>2}. {feat}')
# #
# # # ── Step 3: Export feature metadata for reproducibility ───────────────────────
# # rows = [{
# #     'Model': model,
# #     'Feature_Selection': info['method'],
# #     'n_selected': len(info['features']),
# #     'n_common': len(info['features']),
# #     'n_uncommon': 0,
# #     'common_features': ', '.join(info['features']),
# #     'uncommon_features': '',
# # } for model, info in MODEL_FEATURE_SETS.items()]
# #
# # feature_cmp_df = pd.DataFrame(rows)
# # display(feature_cmp_df)
# # feature_cmp_df.to_csv(OUTPUT_DIR / 'feature_selection_comparison.csv', index=False)
# # with open(OUTPUT_DIR / 'common_features.json', 'w') as f:
# #     json.dump(common_features, f, indent=2)
# # print(f"Saved: {OUTPUT_DIR / 'feature_selection_comparison.csv'}")


<!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed -->

<!-- <!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed --> -->
<!--  -->
<!-- <!-- --- --> -->
<!-- <!-- ## Cell 14 — Model Comparison --> -->


In [ ]:
# ── Original v2 cell (commented out) — restore when needed ──
# # ── Original v2 cell (commented out) — restore when needed ──
# # # ── Step 1: Build comparison table from all model results ───────────────────
# # results_df = pd.DataFrame([v['metrics'] for v in ALL_RESULTS.values()])
# # print('\n=== FULL COMPARISON TABLE ===')
# # display_cols = ['Model', 'Algorithm', 'Feature_Selection', 'AUC', 'Train_AUC', 'AUC_Gap',
# #                 'Precision', 'Recall', 'F1-Score', 'FPR',
# #                 'Train_Time_s', 'Infer_Time_s', 'Peak_Mem_MB', 'Efficiency']
# # display_cols = [c for c in display_cols if c in results_df.columns]
# # print(results_df[display_cols].to_string(index=False))
# #
# # # ── Step 2: Normalize metrics and compute weighted composite score ─────────
# # rank_df = results_df.copy()
# # for col in ['AUC', 'F1-Score', 'Recall', 'Precision', 'Efficiency']:
# #     rank_df[f'{col}_norm'] = rank_df[col] / (rank_df[col].max() + 1e-9)
# # rank_df['FPR_penalty'] = 1 - rank_df['FPR'] / (rank_df['FPR'].max() + 1e-9)
# # rank_df['Time_penalty'] = 1 - rank_df['Train_Time_s'] / (rank_df['Train_Time_s'].max() + 1e-9)
# # rank_df['Mem_penalty']  = 1 - rank_df['Peak_Mem_MB'] / (rank_df['Peak_Mem_MB'].max() + 1e-9)
# #
# # rank_df['Composite_Score'] = (
# #     0.25*rank_df['AUC_norm'] + 0.25*rank_df['F1-Score_norm'] +
# #     0.15*rank_df['Recall_norm'] + 0.15*rank_df['Precision_norm'] +
# #     0.10*rank_df['Efficiency_norm'] + 0.05*rank_df['FPR_penalty'] +
# #     0.03*rank_df['Time_penalty'] + 0.02*rank_df['Mem_penalty']
# # )
# # rank_df = rank_df.sort_values('Composite_Score', ascending=False)
# # best_row = rank_df.iloc[0]
# #
# # # ── Step 3: Show ranking and persist CSV ────────────────────────────────────
# # print('\n=== RANKING (Composite Score) ===')
# # print(rank_df[['Model','Composite_Score','AUC','F1-Score','Train_Time_s','Peak_Mem_MB']]
# #       .to_string(index=False))
# #
# # results_df.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
# # print(f'\nSaved: {OUTPUT_DIR / "model_comparison.csv"}')


In [ ]:
# ── Original v2 cell (commented out) — restore when needed ──
# # ── Original v2 cell (commented out) — restore when needed ──
# # # ── Step 1: Bar chart of classification metrics per algorithm ───────────────
# # fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# #
# # metrics_plot = results_df.set_index('Algorithm')[['AUC','Precision','Recall','F1-Score']]
# # metrics_plot.plot(kind='bar', ax=axes[0,0], rot=0)
# # axes[0,0].set_title('Classification Metrics')
# # axes[0,0].set_ylim(0, 1.05)
# # axes[0,0].legend(loc='lower right')
# #
# # # ── Step 2: ROC curves for each model on held-out test ──────────────────────
# # colors = {'Decision Tree': '#3498db', 'XGBoost': '#e74c3c',
# #           'AdaBoost': '#f39c12', 'Random Forest': '#2ecc71'}
# # for algo, result in ALL_RESULTS.items():
# #     y_prob = result['prob']
# #     if y_prob is not None:
# #         fpr_curve, tpr_curve, _ = roc_curve(y_test, y_prob)
# #         auc_val = roc_auc_score(y_test, y_prob)
# #         axes[0,1].plot(fpr_curve, tpr_curve,
# #                        label=f'{algo} (AUC={auc_val:.3f})', color=colors.get(algo, 'gray'))
# # axes[0,1].plot([0,1],[0,1],'k--', alpha=0.4)
# # axes[0,1].set_title('ROC Curves')
# # axes[0,1].set_xlabel('FPR')
# # axes[0,1].set_ylabel('TPR')
# # axes[0,1].legend(fontsize=8)
# #
# # # ── Step 3: Runtime and memory comparison charts ────────────────────────────
# # results_df.plot(x='Algorithm', y=['Train_Time_s','Infer_Time_s'], kind='bar', ax=axes[1,0], rot=0)
# # axes[1,0].set_title('Training vs Inference Time')
# # axes[1,0].set_ylabel('Seconds')
# #
# # results_df.plot(x='Algorithm', y='Peak_Mem_MB', kind='bar', ax=axes[1,1], rot=0, legend=False, color='coral')
# # axes[1,1].set_title('Peak Memory During Training')
# # axes[1,1].set_ylabel('MB')
# #
# # plt.tight_layout()
# # plt.savefig(OUTPUT_DIR / 'model_comparison_plots.png', dpi=150, bbox_inches='tight')
# # plt.show()


<!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed -->

<!-- <!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed --> -->
<!--  -->
<!-- <!-- --- --> -->
<!-- <!-- ## Cell 15 — Recommended Model & Rationale --> -->


In [ ]:
# ── Original v2 cell (commented out) — restore when needed ──
# # ── Original v2 cell (commented out) — restore when needed ──
# # # ── Step 1: Pick winner by highest composite score ──────────────────────────
# # RECOMMENDED = best_row['Algorithm']
# # print('='*60)
# # print('RECOMMENDED MODEL:', RECOMMENDED)
# # print('='*60)
# # print(f"  Composite Score : {best_row['Composite_Score']:.4f}")
# # print(f"  AUC             : {best_row['AUC']}")
# # print(f"  F1-Score        : {best_row['F1-Score']}")
# # print(f"  Precision       : {best_row['Precision']}")
# # print(f"  Recall          : {best_row['Recall']}")
# # print(f"  Train Time      : {best_row['Train_Time_s']}s")
# # print(f"  Peak Memory     : {best_row['Peak_Mem_MB']} MB")
# # print(f"  Feature Selection: {best_row['Feature_Selection']}")
# #
# # # ── Step 2: Map algorithm name to trained artifact for saving/prediction ──
# # MODEL_MAP = {
# #     'Decision Tree': (dt_model, SELECTED_FEATURES, X_te),
# #     'XGBoost': (xgb_model, SELECTED_FEATURES, X_te),
# #     'AdaBoost': (ada_model, SELECTED_FEATURES, X_te),
# #     'Random Forest': (rf_model, SELECTED_FEATURES, X_te),
# # }
# # final_model, final_features, final_X_test = MODEL_MAP[RECOMMENDED]
# # print(f'\nSelected artifact: {RECOMMENDED} with {len(final_features)} features.')


<!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed -->

<!-- <!-- Original v2 cell preserved below — uncomment when multi-model comparison is needed --> -->
<!--  -->
<!-- <!-- --- --> -->
<!-- <!-- ## Cell 16 — Save Best Model & Predict --> -->


In [ ]:
# ── Original v2 cell (commented out) — restore when needed ──
# # ── Original v2 cell (commented out) — restore when needed ──
# # import joblib
# #
# # # ── Step 1: Persist model and all preprocessing artifacts ─────────────────────
# # joblib.dump(final_model, MODEL_DIR / 'best_model.joblib')
# # joblib.dump(scaler, MODEL_DIR / 'scaler.joblib')
# # joblib.dump(label_encoders, MODEL_DIR / 'label_encoders.joblib')
# # joblib.dump(train_medians, MODEL_DIR / 'train_medians.joblib')
# #
# # # ── Step 2: Save JSON metadata (metrics, features, algorithm name) ──────────
# # meta = {
# #     'algorithm': RECOMMENDED,
# #     'feature_selection': best_row['Feature_Selection'],
# #     'best_unified_fs_method': globals().get('BEST_FS_METHOD'),
# #     'best_unified_fs_features': globals().get('BEST_FS_FEATURES'),
# #     'selected_features': final_features,
# #     'feature_columns': feature_columns,
# #     'metrics': {
# #         'auc': best_row['AUC'],
# #         'precision': best_row['Precision'],
# #         'recall': best_row['Recall'],
# #         'f1_score': best_row['F1-Score'],
# #         'fpr': best_row['FPR'],
# #     },
# #     'model_file': 'best_model.joblib',
# # }
# # with open(MODEL_DIR / 'model_metadata.json', 'w', encoding='utf-8') as f:
# #     json.dump(meta, f, indent=2)
# #
# # print(f'Artifacts saved to {MODEL_DIR}/')
# #
# # # ── Step 3: Run final predictions on held-out test (scored once) ────────────
# # y_final_pred = final_model.predict(final_X_test)
# # y_final_prob = final_model.predict_proba(final_X_test)[:, 1]
# #
# # pred_df = pd.DataFrame({
# #     'actual': y_test.values,
# #     'predicted': y_final_pred,
# #     'probability': y_final_prob,
# #     'actual_label': ['DDoS/DoS' if v else 'Other' for v in y_test.values],
# #     'predicted_label': ['DDoS/DoS' if v else 'Other' for v in y_final_pred],
# # })
# # pred_df.head(20)
# #
# # # ── Download results zip (Colab only) ───────────────────────────────────────
# # try:
# #     from google.colab import files
# #     import shutil
# #     zip_path = '/content/ddos_results'
# #     shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
# #     print('Downloading outputs zip...')
# #     files.download(zip_path + '.zip')
# #     shutil.make_archive('/content/ddos_models', 'zip', MODEL_DIR)
# #     print('Downloading models zip...')
# #     files.download('/content/ddos_models.zip')
# # except ImportError:
# #     print('Not running in Colab — zip download skipped.')
# # except Exception as exc:
# #     print(f'Zip download skipped: {exc}')


---
## Summary — SMOTE Ratio Experiments

| Phase | What it does |
|-------|----------------|
| **Setup (Cells 1–2)** | GPU check, packages, project paths |
| **Data (Cell 3)** | Load from `data/` or optional Kaggle download |
| **EDA & labels (Cells 4–5)** | Overview plots, binary labels |
| **Preprocessing (Cell 6)** | Train-only scaling; train/test split; **no resampling** |
| **Resampling (Cell 8)** | SMOTE or undersampling on **train only** per ratio |
| **Training (Cell 9)** | All models: regularized hyperparams + 90/10 validation early stopping (`patience=25`); XGBoost `scale_pos_weight=1` after SMOTE |
| **Results (Cell 10)** | Per-model Class 0/1 recall, accuracy, F1, AUC per ratio |

**Colab:** Results saved under `/content/outputs/`. Use the download cell at the end.

**Pipeline:** Original dataset → Train/Test split → Resample train → Train model → Evaluate on untouched test.

**Feature selection justification:** See `CICIoMT2024_FeatureSelection_CV_Comparison.ipynb` for train-only CV comparison of Correlation, MI, RFE, and XGB Gain.


---
## Download Results (Colab)

Run after experiments finish to save CSVs and plots to your computer.


In [ ]:
from google.colab import files
import shutil
from pathlib import Path

zip_base = '/content/ddos_outputs'
if Path(OUTPUT_DIR).exists() and any(Path(OUTPUT_DIR).iterdir()):
    shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
    print('Downloading outputs zip...')
    files.download(f'{zip_base}.zip')
else:
    print('No files in outputs yet — run experiment cells first.')
